# Pretraining

**Tags:** training, foundation
**Prerequisites:** None
**Related Concepts:** See flowchart below
**Source:** llm/concepts/pretraining.md

## TL;DR

Unsupervised learning on massive unlabeled text corpus (100B-10T tokens). Model learns to predict next token (causal: GPT-style) or masked tokens (masked: BERT-style). Foundation for all modern LLMs. Results: emergent abilities (reasoning, knowledge, generation). Cost: weeks-months on 100s of GPUs. Trade-off: compute investment now → transfer learning value later.

## Core Intuition

Humans learn language by reading books (not studying grammar rules). Pretraining is similar: model reads billions of tokens and learns patterns—syntax, semantics, facts, reasoning. This compressed knowledge transfers to downstream tasks (classification, generation, QA) via fine-tuning. The goal: learn representations that capture structure of language and world.

## How It Works

**Causal Language Modeling (GPT-style, Autoregressive):**

Objective: predict next token given all previous tokens
```
P(x) = P(x_1) × P(x_2|x_1) × P(x_3|x_1,x_2) × ... × P(x_T|x_1...x_{T-1})

Loss = -log P(x_1...x_T) = Σ_i -log P(x_i | x_{<i})

Example:
  Input: "The cat sat on the"
  Model predicts: "mat" with probability 0.8
  Loss = -log(0.8) = 0.22
```

Architecture:
```
Token embedding → Positional embedding
  ↓
Transformer (12-96 layers)
  Attention (causal mask: attend only to past)
  Feed-forward
  LayerNorm, Dropout
  ↓
Output embedding → softmax → next token probability
```

Key properties:
- **Causal masking:** Q can only attend to positions ≤ Q's position
- **Autoregressive generation:** generate one token at a time (sequential)
- **Harder objective:** predicting future is harder than filling in the middle
- **Better generalization:** learned representations useful for generation

**Masked Language Modeling (BERT-style, Bidirectional):**

Objective: predict masked tokens given full context
```
1. Corrupt text: replace 15% tokens with [MASK]
   "The cat sat on the mat" → "The [MASK] sat on the [MASK]"
2. Model predicts masked tokens bidirectionally
   P(x_2 | context without x_2) = ...
3. Loss: cross-entropy on masked positions only
```

Architecture:
```
Same as causal, but WITHOUT causal masking
Attention: attend to all positions (bidirectional)
```

Key properties:
- **Bidirectional context:** more information for prediction (easier task)
- **Non-autoregressive:** can extract representations in parallel
- **Better for understanding:** classification, retrieval
- **Can't generate directly:** no causal structure for sequential generation

**Scale Laws (Chinchilla / Scaling Laws):**

Performance improves predictably:
```
Loss(N, D) ∝ 1/N^α + 1/D^β

Where:
  N = number of model parameters
  D = number of tokens seen
  α ≈ 1.0, β ≈ 0.14 (empirically)

Scaling law insights:
1. Compute budget C ≈ 6 × N × D (6 FLOPs per parameter-token pair)
2. Optimal: N ≈ D / 20 (Chinchilla ratio)
   Example: 10B token budget → 500M params optimal
3. Training on too much data (D >> C/6N) is inefficient
4. Larger models are more sample-efficient
```

**Training Distribution & Curriculum:**

Data mixing:
```
Wikipedia: 3%
Books: 15%
Code: 25%
Web (Common Crawl): 57%

Different corpora teach different skills:
  - Books: narrative, complex reasoning
  - Code: structured logic, precision
  - Web: diverse topics, common knowledge
  - Wikipedia: factual, encyclopedia-style
```

Curriculum learning (optional):
```
Phase 1: Clean, high-quality data (Books, Wikipedia)
  → establish language foundation
  
Phase 2: Diverse web data
  → learn breadth of topics, styles
  
Phase 3: Domain-specific data (code, scientific papers)
  → specialized knowledge
```

### Workflow Diagram**Note:** This is a basic workflow template. Review and customize based on specific concept.
**Note:** Flowchart diagrams are in the source markdown file (`llm/concepts/{concept}.md`) for better rendering on GitHub.

## Key Properties & Trade-offs

/ Trade-offs

| Aspect | Causal (GPT) | Masked (BERT) | Hybrid |
|--------|--------------|---------------|--------|
| Training objective | Next token prediction | Masked token prediction | Both |
| Directionality | Unidirectional | Bidirectional | Varies |
| Generation | Direct (autoregressive) | Requires decoding tricks | Both |
| Classification | Via prompting/fine-tuning | Direct (embed + classify) | Both |
| Training ease | Harder (future unpredictable) | Easier (context available) | Moderate |
| Data efficiency | Needs more data | Less data needed | Moderate |
| Example models | GPT-2/3/4, Llama, Claude | BERT, RoBERTa | T5, BART |

**Computational Cost:**

Example: Pretraining Llama-2-70B
```
Parameters: 70B
Tokens seen: 2T (2 trillion)
Compute budget: 6 × 70B × 2T = 840 × 10^18 FLOPs

H100 GPU: ~3 × 10^17 FLOPs/s = 1 exaflop/s
Time: 840 × 10^18 FLOPs / 10^18 FLOPs/s = 840s... wait, that's parallel

With 2048 H100 GPUs:
  Total compute: 2048 × 10^18 FLOPs/s = 2 exaflops/s
  Time: 840 × 10^18 / (2 × 10^18) ≈ 420 seconds? No, that's wrong.
  
Correct calculation:
  Time = Compute_FLOPs / (GPUs × FLOPs_per_GPU_per_sec)
  Time = (840 × 10^18) / (2048 × 3 × 10^17) ≈ 1,370 seconds ≈ 23 minutes

This assumes 100% utilization, which is optimistic. In practice:
  - Actual: 2-5 weeks on 2000 H100s
  - Cost: 2-5 weeks × 2000 GPUs × ~$3/hour = $3-7M
```

### Code Implementation

```python
# Key imports for Pretraining
import numpy as np
import torch
from typing import Any

# Pretraining example implementation
class Pretraining:
    """
    Pretraining implementation.
    This is a template - customize with actual code.
    """
    def __init__(self):
        pass

    def process(self, input_data: Any) -> Any:
        # Interview tip: Explain the core insight here
        return input_data
```

### Mathematical Formulation

Include LaTeX equations relevant to this concept.

**Example:**
$$\text{Output} = f(\text{Input})$$

## Related Concepts
**Note:** Flowchart diagrams are in the source markdown file (`llm/concepts/{concept}.md`) for better rendering on GitHub.

### Common Interview Questions

**Q: What is Pretraining used for?**
A: [Add concise answer about practical application]

**Q: What are the main trade-offs of Pretraining?**
A: [Discuss pros/cons and when to use vs alternatives]

**Q: How does Pretraining compare to [related concept]?**
A: [Explain key differences and when to use each]

**Q: What are common mistakes when using Pretraining?**
A: [List 1-2 common pitfalls and how to avoid them]

**Q: Can you explain the intuition behind Pretraining?**
A: [Provide a simple analogy or explanation]

## References

- **Source Document:** `llm/concepts/pretraining.md`
- **Related Papers:** [Add relevant papers]
- **Implementations:**
  - HuggingFace: [Add links]
  - GitHub: [Add links]